In [1]:
import pandas as pd
import seaborn as sns

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 35)

In [2]:
import abcd_helper as abcd

# load conversations data
df_convos = abcd.load_conversations()
df_turns = abcd.load_turns()
print(f"{len(df_convos)} conversations" + f"{len(df_turns)} total turns")

df_schema = abcd.load_intent_slots(as_df=True)
global_slots = set()
df_schema["all_slots"].apply(lambda x: global_slots.update(x))
available_slots = sorted(list(global_slots))

print(f"There are {len(available_slots)} available slots globally")
print(available_slots)

10042 conversations220983 total turns
There are 25 available slots globally
['account_id', 'account_slotval', 'amount', 'change_option', 'company_team', 'customer_name', 'details_slotval', 'email', 'full_address', 'membership_level', 'order_id', 'order_slotval', 'payment_method', 'phone', 'pin_number', 'product', 'reason_slotval', 'refund_target', 'security_answer', 'shipping_option', 'single_item_query', 'storewide_query', 'street_address', 'username', 'zip_code']


In [3]:
# Sampled some conversations for evaluation
df_convos_samples = df_convos.sample(n=10, random_state=42)
print(f'Sampled conversations: {len(df_convos_samples)}')

df_turns_samples = df_turns[df_turns['convo_id'].isin(df_convos_samples['convo_id'].tolist())]
print(f'Sampled messages: {len(df_turns_samples)}')

# prepare data for semantic matcher input
from nlp.context import extract_customer_contexts
all_documents = extract_customer_contexts(df_turns_samples, min_prior_agent_utterances=1, group_by_convo=False)
print(f'Customer messages to be extracted {len(all_documents)}')

Sampled conversations: 10
Sampled messages: 221
Customer messages to be extracted 91


In [4]:
from nlp.slots import SemanticMatcher

BATCH_SIZE = 64
slot_extractor = SemanticMatcher(
    model_name="facebook/bart-large-mnli", 
    hypothesis_template = "The customer specifies their {}",
    threshold=0.7,
    batch_size=BATCH_SIZE
)

/Users/ridho-akbar/Projects/data-science/projects/nlp-dialogue-state-tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 46938.58it/s]


In [5]:
from itertools import batched
from tqdm import tqdm

records = []
for batch_docs in tqdm(list(batched(all_documents, BATCH_SIZE)), desc="Batches"):
    batch_texts = [
        doc.get_full_document(strip_prior_customer_utterances=True)
        for doc in batch_docs
    ]
    batch_inferred = slot_extractor.extract_batch(batch_texts, available_slots, return_scores=True)
    for doc, extracted in zip(batch_docs, batch_inferred):
        inferred_slots = extracted.keys()
        confidence = extracted.values()
        records.append(
            {
                "convo_id": doc.convo_id,
                "turn_id": doc.turn_id,
                "inferred_slots": inferred_slots,
                "confidence_scores": confidence
            }
        )

df_results = pd.DataFrame(records)

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:23<00:00, 11.87s/it]


In [10]:
df_evals = (
    df_results.merge(
        df_turns_samples[['convo_id', 'turn_id', 'slot_tags', 'text']],
        how='left', on=['convo_id', 'turn_id']
    )
)
    
df_evals = df_evals.assign(
    true_slots = df_evals['slot_tags'].map(lambda x: x.split(',') if x==x else [])
)[['convo_id', 'turn_id', 'inferred_slots', 'true_slots', 'text']]

from data_science.evaluation import evaluate_multilabel

eval_result= evaluate_multilabel(
    df_evals, true_col="true_slots", 
    pred_col="inferred_slots", 
    drop_zero_support=False
)

eval_result

Global Metric,Score
exact_match_ratio,52.7%
hamming_loss,0.0448
jaccard_similarity_samples,8.7%
micro_precision,16.5%
micro_recall,95.2%
micro_f1,28.2%
macro_precision,19.4%
macro_recall,32.0%
macro_f1,23.1%
weighted_precision,53.6%


In [7]:
BATCH_SIZE = 64
slot_extractor = SemanticMatcher(
    model_name="facebook/bart-large-mnli", 
    hypothesis_template = "The customer specifies their {}",
    threshold=0.5,
    batch_size=BATCH_SIZE
)

doc = all_documents[3]
test_text = doc.get_full_document(strip_prior_customer_utterances=True)
# test_text = doc.customer_text
# test_text = "my username is andrew123 and I live in London"

true_slots = df_turns[
    (df_turns['convo_id']==doc.convo_id)
    & (df_turns['turn_id']==doc.turn_id)
]['slot_tags'].values.tolist()

extracted = slot_extractor.extract_slots(test_text, available_slots, return_scores=True)

print("Text:\n----------")
print(test_text, "\n----------")
print("Inferred slots:", extracted)
print("True slots:", true_slots)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 32506.16it/s]


Text:
----------
[AGENT]: Thank you Albert - may I also have your account ID and order ID?
[CUSTOMER (CURRENT)]: Account ID D4EHGCEDQD 
----------
Inferred slots: {'account_id': 0.9753134846687317, 'account_slotval': 0.6909916996955872, 'change_option': 0.5207083225250244, 'details_slotval': 0.5540009140968323, 'storewide_query': 0.6152788996696472}
True slots: ['account_id']


In [8]:
list(extracted.keys())

['account_id',
 'account_slotval',
 'change_option',
 'details_slotval',
 'storewide_query']